##### This is a notebook to execute merges for 3 datasets while preserving:
* Data Lineage
* Correctness of what is being merged to what
* Knowing which dataset joins to which(what is the base dataset)

In [1]:
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString
from scipy.spatial import cKDTree

In [2]:
ecoplanet_dataset_path = '../../02_INTERMEDIATE/final_bamboo_parameters_v03.gpkg'
acre_drone_dataset_path = '../../02_INTERMEDIATE/merged_drone_bamboo_params_breast_height_centroids.gpkg'
# acre_height_dataset_path = '../02_INTERMEDIATE/acre_height_validation.gpkg'

In [3]:
ecoplanet_df = gpd.read_file(ecoplanet_dataset_path)
drone_df = gpd.read_file(acre_drone_dataset_path)

In [4]:
drone_df['geometry'].head()

0    POINT (798561.588 9777838.958)
1     POINT (798565.17 9777833.521)
2    POINT (798566.854 9777828.405)
3    POINT (798567.846 9777823.479)
4    POINT (798560.496 9777810.003)
Name: geometry, dtype: geometry

In [6]:
drone_df = gpd.GeoDataFrame(drone_df, geometry="geometry", crs="EPSG:4326")
ecoplanet_df = gpd.GeoDataFrame(
    ecoplanet_df,
    geometry=gpd.GeoSeries.from_wkt(ecoplanet_df['geometry']),
    crs="EPSG:4326",
)

TypeError: Expected bytes or string, got Point

In [5]:
ecoplanet_df.geometry.head()

0     POINT (798563.99 9777833.328)
1     POINT (798561.124 9777837.75)
2     POINT (798566.87 9777828.479)
3    POINT (798563.966 9777833.328)
4     POINT (798567.766 9777823.46)
Name: geometry, dtype: geometry

In [7]:
drone_df.columns

Index(['breast_height_points', 'breast_height_valid_points',
       'breast_height_filtered_points', 'breast_height_cleaning_distance',
       'breast_height_area', 'breast_height_perimeter_m',
       'breast_height_diameter', 'bh_outside_area_m2', 'canopy_diameter',
       'psp_id', 'geometry'],
      dtype='str')

In [8]:
drone_df['fid'] = drone_df.index + 1

In [9]:
ecoplanet_df.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'geometry'],
      dtype='str')

##### Add prefixes to maintain data lineage

In [10]:
ecoplanet_df.columns = [f"ecoplanet_{col}" if col != 'geometry' else col for col in ecoplanet_df.columns]
drone_df.columns = [f"acre_drone_{col}" if col != 'geometry' else col for col in drone_df.columns]

In [11]:
ecoplanet_df.columns

Index(['ecoplanet_psp', 'ecoplanet_year', 'ecoplanet_date_of_collection',
       'ecoplanet_clump_id', 'ecoplanet_total_culms', 'ecoplanet_total_agb',
       'ecoplanet_max_culm_diameter_cm', 'ecoplanet_avg_culm_diameter_cm',
       'ecoplanet_min_culm_diameter_cm', 'ecoplanet_median_culm_diameter_cm',
       'ecoplanet_max_culm_agb', 'ecoplanet_avg_culm_agb',
       'ecoplanet_min_culm_agb', 'ecoplanet_median_culm_agb', 'ecoplanet_hmin',
       'ecoplanet_hmax', 'ecoplanet_hmean', 'ecoplanet_hstd',
       'ecoplanet_hmedian', 'ecoplanet_total_points',
       'ecoplanet_canopy_volume_m3', 'ecoplanet_gap_fraction',
       'ecoplanet_canopy_area', 'ecoplanet_val_height',
       'ecoplanet_val_canopy_diameter', 'geometry'],
      dtype='str')

In [12]:
drone_df.columns

Index(['acre_drone_breast_height_points',
       'acre_drone_breast_height_valid_points',
       'acre_drone_breast_height_filtered_points',
       'acre_drone_breast_height_cleaning_distance',
       'acre_drone_breast_height_area', 'acre_drone_breast_height_perimeter_m',
       'acre_drone_breast_height_diameter', 'acre_drone_bh_outside_area_m2',
       'acre_drone_canopy_diameter', 'acre_drone_psp_id', 'geometry',
       'acre_drone_fid'],
      dtype='str')

##### Local coordinate system to ensure accurate distance matching

In [13]:
crs = 'EPSG:32735'
ecoplanet_df = ecoplanet_df.to_crs(crs)
drone_df = drone_df.to_crs(crs)

In [14]:
drone_df.head()

,acre_drone_breast_height_points,acre_drone_breast_height_valid_points,acre_drone_breast_height_filtered_points,acre_drone_breast_height_cleaning_distance,acre_drone_breast_height_area,acre_drone_breast_height_perimeter_m,acre_drone_breast_height_diameter,acre_drone_bh_outside_area_m2,acre_drone_canopy_diameter,acre_drone_psp_id,geometry,acre_drone_fid
0,3739,46698,1380,0.161555,36.295621,22.555733,6.798016,5.699630,10.241487,PSP-1A-APM-01,POINT (798561.588 9777838.958),1
1,3028,36518,1341,0.152971,26.436323,19.638267,5.801704,6.433417,9.570837,PSP-1A-APM-01,POINT (798565.17 9777833.521),2
2,1002,5810,1002,0.123693,4.092321,7.656746,2.282653,1.133990,3.734397,PSP-1A-APM-01,POINT (798566.854 9777828.405),3
3,1742,13030,1708,0.150000,9.535144,16.407981,3.484325,2.217230,5.622583,PSP-1A-APM-01,POINT (798567.846 9777823.479),4
4,348,1927,348,0.094868,0.803208,3.916569,1.011275,0.921962,2.271427,PSP-1A-APM-01,POINT (798560.496 9777810.003),5


In [15]:
drone_df.isna().sum()

acre_drone_breast_height_points               0
acre_drone_breast_height_valid_points         0
acre_drone_breast_height_filtered_points      0
acre_drone_breast_height_cleaning_distance    0
acre_drone_breast_height_area                 0
acre_drone_breast_height_perimeter_m          0
acre_drone_breast_height_diameter             0
acre_drone_bh_outside_area_m2                 0
acre_drone_canopy_diameter                    0
acre_drone_psp_id                             0
geometry                                      0
acre_drone_fid                                0
dtype: int64

##### Nearest neighbour analysis

In [16]:
# 4. Extract coordinates for fast Nearest-Neighbor matching using a KDTree
owner_coords = list(zip(ecoplanet_df.geometry.x, ecoplanet_df.geometry.y))
drone_coords = list(zip(drone_df.geometry.x, drone_df.geometry.y))
drone_tree = cKDTree(drone_coords)

In [17]:
# Find the single closest drone point for every ecoplanet point
MAX_DISTANCE = 2.0  # 2 meters max cap
distances, indices = drone_tree.query(owner_coords, k=1, distance_upper_bound=MAX_DISTANCE)

In [18]:
 #Build the visual inspection dataset
visual_lines = []

In [19]:
for i, (owner_idx, drone_idx) in enumerate(enumerate(indices)):
    owner_point = ecoplanet_df.geometry.iloc[owner_idx]
    match_distance = distances[owner_idx]
    
    # Check if a valid match was found within 2 meters
    if drone_idx < drone_tree.n and match_distance <= MAX_DISTANCE:
        drone_point = drone_df.geometry.iloc[drone_idx]
        # Create a line connecting the matched clumps
        line = LineString([owner_point, drone_point])
        status = "Matched"
        
        # Pull drone data since it's a valid match
        drone_id = drone_df["acre_drone_fid"].iloc[drone_idx] # Update col name
        drone_area = drone_df["acre_drone_fid"].iloc[drone_idx] # Update col name
    else:
        # If it failed the 2m cap, create a tiny 10cm dummy line pointing north
        # This keeps the unmapped point visible in the line layer for your audit
        line = LineString([owner_point, (owner_point.x, owner_point.y + 0.1)])
        status = "Missed by Drone (No Match)"
        match_distance = np.nan
        drone_id = np.nan
        drone_area = np.nan

    # Combine attributes into the diagnostic record
    line_data = {
        "geometry": line,
        "match_status": status,
        "match_dist_m": match_distance,
        "owner_id": ecoplanet_df["ecoplanet_clump_id"].iloc[owner_idx], # Update col name
        "owner_agb": ecoplanet_df["ecoplanet_total_agb"].iloc[owner_idx],
        "drone_id": drone_id,
        "drone_canopy_area": drone_area
    }
    visual_lines.append(line_data)

# Convert list to a GeoDataFrame
inspection_gdf = gpd.GeoDataFrame(visual_lines, crs=crs)

# 6. Export back to WGS84 for perfect overlay on satellite imagery in QGIS
inspection_gdf.to_crs("EPSG:4326").to_file(f"../../02_INTERMEDIATE/capped_match_inspection.gpkg", driver="GPKG")

print("Inspection file 'capped_match_inspection.gpkg' created successfully with a 2m cap!")


Inspection file 'capped_match_inspection.gpkg' created successfully with a 2m cap!


#### Onto the actual merging of the dataframes

In [20]:
visual_lines

[{'geometry': <LINESTRING (798563.99 9777833.328, 798565.17 9777833.521)>,
  'match_status': 'Matched',
  'match_dist_m': np.float64(1.1959877558843537),
  'owner_id': 'CLUMP #1',
  'owner_agb': np.float64(34.65),
  'drone_id': np.int64(2),
  'drone_canopy_area': np.int64(2)},
 {'geometry': <LINESTRING (798561.124 9777837.75, 798561.588 9777838.958)>,
  'match_status': 'Matched',
  'match_dist_m': np.float64(1.2943155324197764),
  'owner_id': 'CLUMP #1',
  'owner_agb': np.float64(144.28),
  'drone_id': np.int64(1),
  'drone_canopy_area': np.int64(1)},
 {'geometry': <LINESTRING (798566.87 9777828.479, 798566.854 9777828.405)>,
  'match_status': 'Matched',
  'match_dist_m': np.float64(0.07595437432176137),
  'owner_id': 'CLUMP #2',
  'owner_agb': np.float64(16.47),
  'drone_id': np.int64(3),
  'drone_canopy_area': np.int64(3)},
 {'geometry': <LINESTRING (798563.966 9777833.328, 798565.17 9777833.521)>,
  'match_status': 'Matched',
  'match_dist_m': np.float64(1.219590411777907),
  'owner

In [21]:
drone_df.columns

Index(['acre_drone_breast_height_points',
       'acre_drone_breast_height_valid_points',
       'acre_drone_breast_height_filtered_points',
       'acre_drone_breast_height_cleaning_distance',
       'acre_drone_breast_height_area', 'acre_drone_breast_height_perimeter_m',
       'acre_drone_breast_height_diameter', 'acre_drone_bh_outside_area_m2',
       'acre_drone_canopy_diameter', 'acre_drone_psp_id', 'geometry',
       'acre_drone_fid'],
      dtype='str')

In [22]:
ecoplanet_df.columns

Index(['ecoplanet_psp', 'ecoplanet_year', 'ecoplanet_date_of_collection',
       'ecoplanet_clump_id', 'ecoplanet_total_culms', 'ecoplanet_total_agb',
       'ecoplanet_max_culm_diameter_cm', 'ecoplanet_avg_culm_diameter_cm',
       'ecoplanet_min_culm_diameter_cm', 'ecoplanet_median_culm_diameter_cm',
       'ecoplanet_max_culm_agb', 'ecoplanet_avg_culm_agb',
       'ecoplanet_min_culm_agb', 'ecoplanet_median_culm_agb', 'ecoplanet_hmin',
       'ecoplanet_hmax', 'ecoplanet_hmean', 'ecoplanet_hstd',
       'ecoplanet_hmedian', 'ecoplanet_total_points',
       'ecoplanet_canopy_volume_m3', 'ecoplanet_gap_fraction',
       'ecoplanet_canopy_area', 'ecoplanet_val_height',
       'ecoplanet_val_canopy_diameter', 'geometry'],
      dtype='str')

In [23]:
ecoplanet_df = ecoplanet_df.rename(columns={'ecoplanet_psp_id':'ecoplanet_psp'})

In [27]:
# ------------------------------------------------------------------------------
# 1. Base Setup and Drone Column Reset
# ------------------------------------------------------------------------------
final_production_gdf = ecoplanet_df.copy()

# Explicitly set the dataset year column to 2026
# final_production_gdf["year"] = 2026
print(f"Starting merge for {len(final_production_gdf)} EcoPlanet clumps for Year 2026...")
# Clean up prefixes from the base dataset (e.g., 'ecoplanet_height' -> 'height')
final_production_gdf.columns = [
    col.replace("ecoplanet_", "") for col in final_production_gdf.columns
]

# Define the precise drone metrics columns you want to replace
drone_cols_to_pull = [
    'breast_height_points', 'breast_height_valid_points',
       'breast_height_filtered_points', 'breast_height_cleaning_distance',
       'breast_height_area', 'breast_height_perimeter_m',
       'breast_height_diameter', 'bh_outside_area_m2'
]
# Clean up the column headers of your incoming new drone dataframe to match
drone_df.columns = [
    col.replace("acre_drone_", "") for col in drone_df.columns
]
# CRITICAL TWEAK: Drop any existing versions of these drone columns to avoid conflicts,
# while keeping all other anchor data columns completely untouched.
existing_drone_cols = [col for col in drone_cols_to_pull if col in final_production_gdf.columns]
if existing_drone_cols:
    final_production_gdf = final_production_gdf.drop(columns=existing_drone_cols)
    print(f"Dropped {len(existing_drone_cols)} old drone metric columns for clean replacement.")

# Initialize the fresh drone columns as object type to prevent TypeErrors during assignment
for col in drone_cols_to_pull:
    final_production_gdf[col] = None

# ------------------------------------------------------------------------------
# 2. Extract and assign NEW drone metrics using your verified inspection indices
# ------------------------------------------------------------------------------
# Generate your coordinates from the anchor dataframe for KDTree querying
owner_coords = list(zip(final_production_gdf.geometry.x, final_production_gdf.geometry.y))

# Assuming drone_df contains your newly re-done drone metrics/centroids
drone_coords = list(zip(drone_df.geometry.x, drone_df.geometry.y))
drone_tree = cKDTree(drone_coords)
distances, indices = drone_tree.query(owner_coords, k=1, distance_upper_bound=MAX_DISTANCE)

# Mask out pairings that exceed your distance thresholds or index bounds
drone_match_mask = (indices < drone_tree.n) & (distances <= MAX_DISTANCE)

# Vectorized transfer of matched new drone records
if drone_match_mask.any():
    matched_drone_indices = indices[drone_match_mask]
    for col in drone_cols_to_pull:
        final_production_gdf.loc[drone_match_mask, col] = drone_df[col].iloc[matched_drone_indices].values

# Add or overwrite tracking column to separate matched clumps from missed drone ones
final_production_gdf["drone_match_status"] = np.where(drone_match_mask, "Matched", "Missed by Drone")

# ------------------------------------------------------------------------------
# 4. Verification Check & Export
# ------------------------------------------------------------------------------
# Print out columns to visually verify the spatial append worked perfectly
print("\n--- Verified Output Columns ---")
validation_cols_created = [col for col in final_production_gdf.columns if col.startswith('val_')]
print(f"Total columns in dataset: {len(final_production_gdf.columns)}")
print(f"Created/Retained {len(validation_cols_created)} validation columns: {validation_cols_created}")

output_production_path = "../../02_INTERMEDIATE/final_bamboo_parameters_v09.gpkg"
final_production_gdf.to_file(output_production_path, driver="GPKG")

print("\n--- Final Merge Summary ---")
print(f"Master file saved successfully to: {output_production_path}")
print(f"Total anchor EcoPlanet clumps: {len(final_production_gdf)}")
print(f"Total clumps paired with drone metrics: {drone_match_mask.sum()}")


Starting merge for 225 EcoPlanet clumps for Year 2026...

--- Verified Output Columns ---
Total columns in dataset: 35
Created/Retained 2 validation columns: ['val_height', 'val_canopy_diameter']

--- Final Merge Summary ---
Master file saved successfully to: ../../02_INTERMEDIATE/final_bamboo_parameters_v09.gpkg
Total anchor EcoPlanet clumps: 225
Total clumps paired with drone metrics: 173


In [28]:
final_production_gdf.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'geometry', 'breast_height_points', 'breast_height_valid_points',
       'breast_height_filtered_points', 'breast_height_cleaning_distance',
       'breast_height_area', 'breast_height_perimeter_m',
       'breast_height_diameter', 'bh_outside_area_m2', 'drone_match_status'],
      dtype='str')

In [29]:
len(final_production_gdf)

225

In [31]:
final_production_gdf = final_production_gdf[['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms', 'total_agb',
       'max_culm_diameter_cm', 'avg_culm_diameter_cm', 'min_culm_diameter_cm',
       'median_culm_diameter_cm', 'max_culm_agb', 'avg_culm_agb',
       'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax', 'hmean', 'hstd', 'hmedian', 'total_points',
       'canopy_volume_m3', 'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter', 'breast_height_points', 'breast_height_valid_points',
       'breast_height_filtered_points', 'breast_height_cleaning_distance',
       'breast_height_area', 'breast_height_perimeter_m',
       'breast_height_diameter', 'bh_outside_area_m2', 'geometry']]

In [32]:
final_production_gdf.to_file(output_production_path, driver="GPKG")